In [30]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression


In [31]:
# Configuration for project

pd.set_option('display.max_columns',None)
pd.set_option('display.float_format',lambda x:f"{x: .3f}")
sns.set_theme(style='darkgrid')
plt.rcParams.update({
    "axes.titlesize":10,
    "axes.labelsize":9,
    "xtick.labelsize":8,
    "ytick.labelsize":8,
})

RANDOM_STATE=42
DATASET_PATH="../data/customer_churn.csv"
TARGET_COLUMN="Churn Value"

In [32]:
df = pd.read_csv(DATASET_PATH, engine="python")

In [33]:
print("DataFrame",df.shape)

DataFrame (7043, 33)


In [34]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964,-118.273,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.850,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059,-118.307,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.700,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048,-118.294,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.650,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062,-118.316,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.800,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039,-118.266,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.700,5036.3,Yes,1,89,5340,Competitor had better devices


In [35]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [37]:
df = df.drop(columns=[
    'CustomerID',
    'Count',
    'Country',
    'State',
    'Lat Long',
    'Churn Score',
    'Churn Reason'
])

In [38]:
df

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,CLTV
0,Los Angeles,90003,33.964,-118.273,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.850,108.15,Yes,1,3239
1,Los Angeles,90005,34.059,-118.307,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.700,151.65,Yes,1,2701
2,Los Angeles,90006,34.048,-118.294,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.650,820.5,Yes,1,5372
3,Los Angeles,90010,34.062,-118.316,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.800,3046.05,Yes,1,5003
4,Los Angeles,90015,34.039,-118.266,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.700,5036.3,Yes,1,5340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Landers,92285,34.342,-116.539,Female,No,No,No,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.150,1419.4,No,0,5306
7039,Adelanto,92301,34.668,-117.536,Male,No,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.800,1990.5,No,0,2140
7040,Amboy,92304,34.560,-115.637,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.200,7362.9,No,0,5560
7041,Angelus Oaks,92305,34.168,-116.864,Female,No,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.600,346.45,No,0,2793


In [39]:
df.shape

(7043, 26)

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7043 non-null   str    
 1   Zip Code           7043 non-null   int64  
 2   Latitude           7043 non-null   float64
 3   Longitude          7043 non-null   float64
 4   Gender             7043 non-null   str    
 5   Senior Citizen     7043 non-null   str    
 6   Partner            7043 non-null   str    
 7   Dependents         7043 non-null   str    
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   str    
 10  Multiple Lines     7043 non-null   str    
 11  Internet Service   7043 non-null   str    
 12  Online Security    7043 non-null   str    
 13  Online Backup      7043 non-null   str    
 14  Device Protection  7043 non-null   str    
 15  Tech Support       7043 non-null   str    
 16  Streaming TV       7043 non-null   

In [41]:
num_cols=df.select_dtypes(include=[np.number]).columns.tolist()
cat_columns=df.select_dtypes(include=['object']).columns.tolist()

print("target column:",TARGET_COLUMN)
print("Numerical Columns",num_cols)
print("Categorical Columns",cat_columns)

target column: Churn Value
Numerical Columns ['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Churn Value', 'CLTV']
Categorical Columns ['City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Total Charges', 'Churn Label']


/tmp/ipykernel_24754/3275882909.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns=df.select_dtypes(include=['object']).columns.tolist()


In [42]:
# Checking Missing Values

print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
City                 0
Zip Code             0
Latitude             0
Longitude            0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn Label          0
Churn Value          0
CLTV                 0
dtype: int64


In [43]:
# To check that how many columns have frequent common values top 20

for col in df.columns:
    print(df[col].value_counts().head(20))

City
Los Angeles       305
San Diego         150
San Jose          112
Sacramento        108
San Francisco     104
Fresno             64
Long Beach         60
Oakland            52
Stockton           44
Glendale           40
Bakersfield        40
Riverside          32
Berkeley           32
Whittier           30
Pasadena           30
San Bernardino     28
Irvine             28
Anaheim            28
Santa Barbara      28
Modesto            28
Name: count, dtype: int64
Zip Code
90003    5
90005    5
90006    5
90010    5
90015    5
90020    5
90022    5
90024    5
90028    5
90029    5
90032    5
90039    5
90041    5
90042    5
90056    5
90061    5
90063    5
90065    5
90211    5
90255    5
Name: count, dtype: int64
Latitude
33.964    5
34.059    5
34.048    5
34.062    5
34.039    5
34.066    5
34.024    5
34.066    5
34.100    5
34.090    5
34.079    5
34.111    5
34.137    5
34.116    5
33.988    5
33.921    5
34.044    5
34.109    5
34.064    5
33.978    5
Name: count, dtype: int64

In [44]:
duplicated_set= df.duplicated()
num_duplicates = duplicated_set.sum()

print("Number of duplicate rows:", num_duplicates)

Number of duplicate rows: 0


In [45]:
# drop duplicate value

df = df.drop_duplicates()
print("Shape after dropping duplicate values:", df.shape)

Shape after dropping duplicate values: (7043, 26)


In [47]:
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Zip Code,7043.000,93521.965,1865.795,90001.000,92102.000,93552.000,95351.000,96161.000
Latitude,7043.000,36.282,2.456,32.556,34.031,36.392,38.225,41.962
Longitude,7043.000,-119.799,2.158,-124.301,-121.815,-119.731,-118.043,-114.193
Tenure Months,7043.000,32.371,24.559,0.000,9.000,29.000,55.000,72.000
Monthly Charges,7043.000,64.762,30.090,18.250,35.500,70.350,89.850,118.750
Churn Value,7043.000,0.265,0.442,0.000,0.000,0.000,1.000,1.000
CLTV,7043.000,4400.296,1183.057,2003.000,3469.000,4527.000,5380.500,6500.000
